In [ ]:
# Jupyter Notebook to run ORION on DAS data (example from Azuma Volcano)
#Plots are saved in the output directory and not shown in the notebook

In [ ]:
#import necessary libraries 
import os
import glob
from tqdm import tqdm
from orionidas import DASChannelSelector 

/home/emanuele/programs/anaconda3/envs/phd_24/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
# ## Configuration Paths

# Fiber geometry path and simplified geometry path 
geometry_csv_path = '/home/emanuele/post_doc/ORION/safe/azuma_volcano/AZUMA_VOLCANO_GEOM_new.txt'
simplified_geometry_csv = '/home/emanuele/post_doc/ORION/safe/azuma_volcano/simplified_geometry_azuma_volcano.csv'

# Directory containing DAS files
das_directory = '/home/emanuele/post_doc/ORION/safe/azuma_volcano'

# Output paths
clustering_save_path = '/home/emanuele/post_doc/ORION/safe/clustering_results_azuma_volcano.pkl'
output_root = '/home/emanuele/post_doc/ORION/safe/output_orion_azuma_volcano'


In [ ]:
# ## File Patterns

mseed_pattern = os.path.join(das_directory, 'noa2024*_das.mseed')
h5_pattern = os.path.join(das_directory, '7M_S_*.h5')
npy_pattern = os.path.join(das_directory, 'AZUMA_DAS_JAPAN_20190704_123410_123510.npy') #just this one works in this case


In [ ]:
# ## Find Files

mseed_files = sorted(glob.glob(mseed_pattern))
h5_files = sorted(glob.glob(h5_pattern))
npy_files = sorted(glob.glob(npy_pattern))


In [ ]:
# ## Build Name/Path Pairs

mseed_names_and_paths = [(os.path.basename(f).replace('noa2024','').replace('_das.mseed',''), f) for f in mseed_files]
h5_names_and_paths = [(os.path.splitext(os.path.basename(f))[0], f) for f in h5_files]
npy_names_and_paths = [(os.path.splitext(os.path.basename(f))[0], f) for f in npy_files]

names_and_paths = mseed_names_and_paths + h5_names_and_paths + npy_names_and_paths


In [6]:
# %%
def spatial_clustering(selector, simplified_geometry_csv, clustering_save_path):
    steps = [
        ("Simplify geometry CSV", lambda: selector.simplify_geometry_csv(simplified_geometry_csv)),
        ("Load geometry", lambda: selector.load_geometry(simplified_geometry_csv)),
        ("Plot geometry", selector.plot_geometry),
        ("Compute average azimuth", lambda: selector.compute_average_azimuth(chunk_size=10)),
        ("Estimate DBSCAN eps", lambda: setattr(selector, 'eps_est', selector.estimate_dbscan_eps(min_samples=10))),
        ("Cluster by azimuth distance", lambda: selector.cluster_by_azimuth_distance(eps=selector.eps_est, min_samples=10, window_size=10)),
        ("Plot geometry with clusters", selector.plot_geometry_with_clusters),
        ("Save clustering", lambda: selector.save_clustering(clustering_save_path)),
    ]
    
    for desc, func in tqdm(steps, desc="Spatial clustering steps"):
        func()


In [7]:
# %%
for i, (name, das_data_path) in enumerate(tqdm(names_and_paths, desc="Processing datasets")):
    print(f"\nProcessing: {name}")
    output_dir = os.path.join(output_root, name)

    selector = DASChannelSelector(
        name=name,
        geometry_path=geometry_csv_path,
        das_data_path=das_data_path,
        output_dir=output_dir
    )

    if i == 0:
        spatial_clustering(selector, simplified_geometry_csv, clustering_save_path)
    else:
        selector.load_clustering(clustering_save_path)
        if not hasattr(selector, 'chunk_centers') or selector.chunk_centers is None:
            selector.compute_chunk_centers()

    # Read DAS data
    selector.read_das_data(
        sampling_rate=200, 
        detrend_data=True,
        apply_filter=True,
        apply_taper=True,
        taper_alpha=0.01,
        low_freq=3,
        high_freq=20,
        dx=2, 
        remove_k0=True
    )

    # Analyze waveforms and select best channels
    selector.analyze_waveforms_and_select_best(
        gauge_length=40,    
        subsection_size=int(selector.avg_cluster_size),
        win_snr=2,            
        step_snr=0.5, 
        win=4,
        start_event=None,
        percentile=1, 
        sta_window_sec=0.5,
        lta_window_sec=5.0,
        sta_lta_on=3.5,
        sta_lta_off=1.2,          
        snr_score_weight=1,
        coherence_score_weight=1,
        noise_rms_weight=1,
        n_final_select=50,
        min_channel_distance=10
    )

    # Save and plot results
    selector.save_selected_traces_to_mseed(save_non_selected=False, n_subset_orion=50)
    selector.load_selected_traces_from_mseed_and_plot()
    selector.plot_selected_traces_on_data()


Processing datasets:   0%|          | 0/1 [00:00<?, ?it/s]


Processing: AZUMA_DAS_JAPAN_20190704_123410_123510


Simplified geometry saved to /home/emanuele/post_doc/ORION/safe/azuma_volcano/simplified_geometry_azuma_volcano.csv


Chunks: 1355, Avg azimuth: 359.05°
Estimating eps using k-distance heuristic...


Estimated eps for DBSCAN: 0.473
Contiguous DBSCAN completed.
Number of clusters: 24
Number of noise points: 135
Average number of channels per cluster: 30.50


...

Section select: |████████████████████████████████████████| 100.0% Complete
Saved full selected (stacked) traces: 49 traces
Saved final selected (stacked) traces: 49 traces
Saved full uniform (unstacked) traces: 49 traces to '/home/emanuele/post_doc/ORION/safe/output_orion_azuma_volcano/AZUMA_DAS_JAPAN_20190704_123410_123510/selected_traces'


No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


Plot saved to: /home/emanuele/post_doc/ORION/safe/output_orion_azuma_volcano/AZUMA_DAS_JAPAN_20190704_123410_123510/AZUMA_DAS_JAPAN_20190704_123410_123510_stack_vs_individual_sta_lta_plot.pdf
Processed and plotted 49 stacked traces from '/home/emanuele/post_doc/ORION/safe/output_orion_azuma_volcano/AZUMA_DAS_JAPAN_20190704_123410_123510'
Plot saved to: /home/emanuele/post_doc/ORION/safe/output_orion_azuma_volcano/AZUMA_DAS_JAPAN_20190704_123410_123510/AZUMA_DAS_JAPAN_20190704_123410_123510_das_data.pdf
Skipping invalid line: section_id	das_id	snr	coherence	rms	combined


Processing datasets: 100%|██████████| 1/1 [00:41<00:00, 41.58s/it]

Plot saved to: /home/emanuele/post_doc/ORION/safe/output_orion_azuma_volcano/AZUMA_DAS_JAPAN_20190704_123410_123510/AZUMA_DAS_JAPAN_20190704_123410_123510_automatic_channel_selection_with_selected_traces.pdf
